# From screenplay to character networks
<a href="https://colab.research.google.com/github/KatherineElkins/programminghumanity2026/blob/main/barbie_network_analysis_2026.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

Run from the top. **Upload → review names/scenes → count relationships → compare networks → remove a character.** No CSV or AI extraction is required beforehand.

The Barbie demonstration now counts directly from the course screenplay PDF. We count **shared-scene speakers**, not every silent character present. Direction means **consecutive speaking turns**, not a verified addressee. Both definitions are choices to explain in your own project.


In [ ]:
# One PDF-reading dependency; no language-model token is needed.
%pip -q install pymupdf
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import io, json, urllib.request, zipfile
from pathlib import Path


## Before running: make four predictions
1. Who will have the most partners? Why?
2. Who will bridge otherwise separate groups?
3. Who may be present but quiet—or speak often to very few people?
4. What communities do you expect?

Save your predictions. Return to them after seeing the results.


## 1. Extract scene headings and speaker cues
Leave `UPLOAD_SCREENPLAY = False` for Barbie. For your own PDF or UTF-8 text file, set it to True. Scanned PDFs need OCR first. Screenplay layouts differ: inspect the scene and name lists rather than assuming extraction is correct.


In [ ]:
"""Count screenplay speaker co-occurrence and turn transitions, with evidence.

These are operational relationships, not inferred intentions or addressees.
PDF layout varies: review candidate speaker names and scene headings before use.
"""
import re
import collections
import itertools
import hashlib
import pandas as pd

SCENE = re.compile(r'^\s*(?:\d+\s+)?(?:INT\.?\s*[/.-]?\s*EXT\.?|EXT\.?\s*[/.-]?\s*INT\.?|INT\.|EXT\.|I/E\.)', re.I)

def clean_cue(text):
    text = re.sub(r'\([^)]*\)', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def pdf_lines(data):
    import fitz
    doc = fitz.open(stream=data, filetype='pdf')
    rows = []
    for page_index, page in enumerate(doc):
        seen = set()
        for block in page.get_text('dict')['blocks']:
            for line in block.get('lines', []):
                text = ''.join(s['text'] for s in line['spans']).strip()
                x, y = line['bbox'][:2]
                key = (round(x, 1), round(y, 1), text)
                if not text or key in seen or y < 45 or y > page.rect.height - 40:
                    continue
                seen.add(key)
                rows.append({'page': page_index + 1, 'x': x, 'y': y,
                             'width': page.rect.width, 'text': text})
    if not rows:
        raise ValueError('No selectable text found. OCR a scanned PDF or upload text.')
    candidates = [r for r in rows if r['text'].isupper()
                  and re.search('[A-Z]', r['text'])
                  and not r['text'].startswith('(')
                  and not SCENE.match(r['text'])
                  and .27 < r['x'] / r['width'] < .66]
    if not candidates:
        raise ValueError('No speaker column detected; try a text screenplay and review its labels.')
    speaker_x = collections.Counter(round(r['x'] / r['width'], 2) for r in candidates).most_common(1)[0][0]
    for r in rows:
        # Also include split-column dialogue cues; candidate names remain reviewable.
        r['cue_candidate'] = (r['text'].isupper() and re.search('[A-Z]', r['text'])
             and not r['text'].startswith('(') and not SCENE.match(r['text'])
             and (abs(r['x'] / r['width'] - speaker_x) < .025
                  or abs(r['x'] / r['width'] - .27) < .018
                  or abs(r['x'] / r['width'] - .635) < .025))
    return sorted(rows, key=lambda r: (r['page'], round(r['y'], 1), r['x']))

def text_lines(data):
    text = data.decode('utf-8-sig') if isinstance(data, bytes) else data
    rows = []
    for index, line in enumerate(text.splitlines()):
        text = line.strip()
        if not text:
            continue
        rows.append({'page': 1, 'x': 0, 'y': index, 'width': 1, 'text': text,
                     'cue_candidate': bool(text.isupper() and len(text) <= 55
                          and re.search('[A-Z]', text) and not text.startswith('(')
                          and not SCENE.match(text))})
    return rows

def extract_cues(data, filename, extra_headings=()):
    lines = pdf_lines(data) if filename.lower().endswith('.pdf') else text_lines(data)
    events, scenes = [], []
    scene_id = 0
    heading = 'Before first detected scene'
    for r in lines:
        text = r['text']
        is_heading = bool(SCENE.match(text) or text in set(extra_headings))
        if is_heading:
            scene_id += 1
            heading = text
            scenes.append({'scene': scene_id, 'heading': heading, 'pdf_page': r['page']})
        elif r['cue_candidate'] and scene_id:
            events.append({'event': len(events)+1, 'scene': scene_id, 'heading': heading,
                           'pdf_page': r['page'], 'y': round(r['y'], 1),
                           'raw_cue': text, 'speaker': clean_cue(text),
                           'voice_over': bool(re.search(r'\(V\.?O\.?\)', text, re.I)),
                           'continued': 'CONT' in text and '(' in text})
    if not scenes or not events:
        raise ValueError('Scenes/speakers not found. Review the screenplay format; do not treat an empty extraction as a network.')
    return pd.DataFrame(events), pd.DataFrame(scenes)

def count_networks(cues, aliases, included):
    included = set(included)
    cues = cues.copy()
    cues['character'] = cues.speaker.map(lambda s: aliases.get(s, s))
    cues['included'] = cues.character.isin(included) & ~cues.voice_over
    selected = cues[cues.included].copy()
    # Co-occurrence: each unordered pair gets ONE count per scene.
    co_rows = []
    for scene, group in selected.groupby('scene'):
        for a, b in itertools.combinations(sorted(set(group.character)), 2):
            co_rows.append({'Source': a, 'Target': b, 'scene': scene,
                            'heading': group.iloc[0].heading,
                            'pdf_pages': ','.join(map(str, sorted(set(group.pdf_page))))})
    co_audit = pd.DataFrame(co_rows, columns=['Source','Target','scene','heading','pdf_pages'])
    # Consecutive cue transitions; NOT addressee inference. No crossing a scene.
    turn_rows = []
    for scene, group in cues.groupby('scene', sort=False):
        previous = None
        for (page, y), simultaneous in group.groupby(['pdf_page', 'y'], sort=False):
            if len(simultaneous) > 1:
                previous = None  # Side-by-side cues do not establish temporal order.
                continue
            row = simultaneous.iloc[0]
            if not row.included:
                previous = None  # Do not jump across an excluded speaker.
                continue
            if row.continued and previous is not None and previous.character == row.character:
                continue
            if previous is not None and previous.character != row.character:
                turn_rows.append({'Source': previous.character, 'Target': row.character,
                    'scene': scene, 'heading': row.heading, 'pdf_page': row.pdf_page,
                    'from_event': previous.event, 'to_event': row.event})
            previous = row
    turn_audit = pd.DataFrame(turn_rows, columns=['Source','Target','scene','heading','pdf_page','from_event','to_event'])
    def aggregate(frame):
        if frame.empty:
            return pd.DataFrame(columns=['Source','Target','Weight'])
        return frame.groupby(['Source','Target']).size().reset_index(name='Weight')
    return aggregate(co_audit), aggregate(turn_audit), co_audit, turn_audit, cues


In [ ]:
UPLOAD_SCREENPLAY = False
if UPLOAD_SCREENPLAY:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload one PDF or text screenplay.')
    source_name, source_bytes = next(iter(uploaded.items()))
    extra_headings = []
    initial_aliases = {}
    excluded_names = set()
else:
    source_name = 'Barbie-Read-The-Screenplay.pdf'
    source_url = 'https://raw.githubusercontent.com/KatherineElkins/networkx_miniproject_3_programming-humanity_2025/main/Barbie-Read-The-Screenplay.pdf'
    source_bytes = urllib.request.urlopen(source_url).read()
    # Additional printed subheadings checked in this particular screenplay.
    extra_headings = ['LATER THAT EVENING...', 'KITCHEN', 'QUICK IMPOSSIBLE MONTAGE:', 'BACK TO MATTEL:', 'TRANSPORTATION MONTAGE', 'CUT BACK TO: VENICE BEACH. REAL WORLD', 'CUT TO: THEATRE']
    initial_aliases = {'BARBIE': 'BARBIE MARGOT', '1950 WOMAN': 'RUTH', 'A VOICE': 'RUTH', 'A VOICE ON THE WIND': 'RUTH', 'RECEPTIONIST/GLORIA': 'GLORIA', 'POLICE MAN': 'POLICEMAN', 'POLICE MAN #2': 'POLICEMAN #2', 'THE OTHER GIRL': 'OTHER GIRL', 'HELLEN MIRREN': 'HELEN MIRREN'}
    excluded_names = set(['HELEN MIRREN', 'ADVERTISING VOICE', 'APP', 'RADIO', 'MATCHBOX 20', 'BARBIES', 'BARBIES/KENS', 'BARBIE ASTRONAUTS', 'ALL BARBIES AND KENS', 'BUSINESSMEN', 'CONSTRUCTION WORKERS', 'KENS', 'ALL THE KENS', 'ALL THE KENS DOWN BELOW', 'ALL OF THE EXECUTIVES', 'EXECUTIVES ALL TOGETHER', 'EXECUTIVES IN UNISON', '1/2 EXECUTIVES ALL TOGETHER', 'THE OTHER HALF ALL TOGETHER', 'GLORIA AND SASHA'])

cues, scenes = extract_cues(source_bytes, source_name, extra_headings)
print(f'{len(scenes)} detected scene sections; {len(cues)} candidate speaker cues.')
print('Preview scene boundaries. A montage/subheading may need separate treatment.')
display(scenes.head(15))
name_counts = cues.groupby('speaker').size().sort_values(ascending=False)
display(name_counts.rename('Cue count').to_frame())


## 2. Review before counting
Merge aliases only when they are the same character. Use Ctrl/Command-click to toggle names. Choose the individual characters to include; exclude narrator, collective voices, and layout mistakes as appropriate. Barbie's supplied aliases were checked in context (for example, the 1950 woman identifies herself as Ruth).

A pair earns **one shared-scene count** if both selected characters have a non-voice-over speaker cue in that scene. Silent appearances are omitted. A directed edge A → B counts B's cue immediately following A's within the scene. Simultaneous cues and excluded speakers break the sequence. These are **turn transitions**, not inferred conversations.

If a scene heading is missed, add its exact printed text to `extra_headings` above and rerun extraction. The seven Barbie additions are documented in that cell. Read the audit for at least three pairs before interpreting results.


In [ ]:
alias_box = widgets.Textarea(value=json.dumps(initial_aliases, indent=2),
    description='Aliases:', layout=widgets.Layout(width='95%', height='180px'),
    style={'description_width': 'initial'})
characters = sorted(set(cues.speaker))
include_box = widgets.SelectMultiple(options=characters,
    value=tuple(s for s in characters if initial_aliases.get(s, s) not in excluded_names),
    description='Include:', rows=12, layout=widgets.Layout(width='95%'))
review_button = widgets.Button(description='Build reviewed networks', button_style='primary')
review_output = widgets.Output()
def build_reviewed(_):
    global df_copresence, df_dialogue, G, G_dialogue, co_audit, turn_audit, cue_audit, aliases
    with review_output:
        clear_output(wait=True)
        aliases = json.loads(alias_box.value)
        included = [aliases.get(s, s) for s in include_box.value]
        df_copresence, df_dialogue, co_audit, turn_audit, cue_audit = count_networks(cues, aliases, included)
        if df_copresence.empty or df_dialogue.empty:
            raise ValueError('No relationships found. Review names, aliases, and scene boundaries.')
        G = nx.from_pandas_edgelist(df_copresence, 'Source', 'Target', edge_attr='Weight')
        G_dialogue = nx.from_pandas_edgelist(df_dialogue, 'Source', 'Target',
                                         edge_attr='Weight', create_using=nx.DiGraph())
        actual_characters = sorted(set(cue_audit.loc[cue_audit.included, 'character']))
        G.add_nodes_from(actual_characters)
        G_dialogue.add_nodes_from(actual_characters)
        print(f'Shared-scene speakers: {len(G)} nodes, {G.number_of_edges()} pairs; '
              f'turn transitions: {len(G_dialogue)} nodes, {G_dialogue.number_of_edges()} directed pairs.')
        display(df_copresence.sort_values('Weight', ascending=False).head(10))
        print('Source scenes for the strongest pair:')
        pair = df_copresence.sort_values('Weight', ascending=False).iloc[0]
        display(co_audit[(co_audit.Source == pair.Source) & (co_audit.Target == pair.Target)])
        print('Changed the review? Rerun the analysis cells below to refresh their graphs.')
review_button.on_click(build_reviewed)
display(alias_box, include_box, review_button, review_output)
build_reviewed(None)


## Restore presence and actual addressees
The automatic first pass counts **co-speakers** and **speaker transitions**. For last year's questions, review **physical presence** (including silent characters) and **actual addressees** instead. A person mentioned is not necessarily present; the next speaker is not necessarily the addressee.

The next cell creates an editable scene file and a speech-review template. Review the screenplay or use this extraction prompt with an AI tool, then check its evidence:

> For each scene, list all named characters physically present, including silent characters and entrances/exits. Exclude mere mentions. Give a page reference and brief evidence for each person. Separately list each speech, its speaker, and its actual addressee(s), with evidence. Give each speech a unique ID. Mark unclear addressees as uncertain; do not substitute the next speaker. Mark soliloquies, asides, and group addresses explicitly. Do not invent names or resolve ambiguity silently.

Check 2–3 familiar scenes for presence and at least one scene speech by speech. Use consistent names. For a group address, list only individually supported addressees; leave unresolved targets out and document them. Count a speech once per supported addressee. Review all speeches within your chosen scene scope, including records with no interpersonal addressee; spot-checking does not establish complete coverage. The templates remain **unreviewed** until you mark them reviewed.

Edit the JSON files in Colab's Files sidebar; rerun the next two cells. You can review a scene subset first, but both networks must use that same subset. Set `USE_REVIEWED = True` to analyze reviewed relationships.


In [ ]:
presence_path = Path('presence_review.json')
speech_path = Path('speech_review.json')
if not presence_path.exists():
    scaffold = []
    for row in scenes.itertuples():
        names = sorted(set(cue_audit.loc[(cue_audit.scene == row.scene) & cue_audit.included, 'character']))
        scaffold.append({'scene': int(row.scene), 'heading': row.heading, 'reviewed': False,
            'present': [{'character': name, 'evidence': ''} for name in names],
            'source_reference': f'PDF page {row.pdf_page}', 'notes': ''})
    presence_path.write_text(json.dumps(scaffold, indent=2))
if not speech_path.exists():
    speech_path.write_text('[]')
print('Edit presence_review.json and speech_review.json in the Files sidebar.')
print('Speech record format (example schema, not screenplay data):')
print(json.dumps({'scene': 1, 'speech_id': 'unique ID', 'speaker': 'NAME',
    'addressees': ['NAME'], 'kind': 'direct', 'reviewed': False,
    'source_reference': 'PDF page / speech', 'evidence': 'supporting text',
    'notes': 'Record uncertain, group, aside or soliloquy decisions here.'}, indent=2))


In [ ]:
USE_REVIEWED = False
relationship_mode = 'Automatic proxies: co-speakers / speaker transitions'
reviewed_presence = json.loads(presence_path.read_text())
reviewed_speeches = json.loads(speech_path.read_text())
if USE_REVIEWED:
    from itertools import combinations
    selected_scenes = [r for r in reviewed_presence if r.get('reviewed') is True]
    if not selected_scenes:
        raise ValueError('Review scenes and mark reviewed: true before using presence data.')
    scene_ids = [r['scene'] for r in selected_scenes]
    if len(scene_ids) != len(set(scene_ids)):
        raise ValueError('Duplicate scene IDs in presence review.')
    presence_rows, address_rows, roster = [], [], set()
    for r in selected_scenes:
        if not r.get('source_reference') or any(not x.get('evidence') for x in r['present']):
            raise ValueError('Each reviewed presence needs evidence and a scene source reference.')
        names = sorted(set(x['character'] for x in r['present']))
        roster.update(names)
        for a, b in combinations(names, 2):
            presence_rows.append({'Source': a, 'Target': b, 'scene': r['scene'],
                'source_reference': r['source_reference']})
    speech_ids = set()
    accepted = [r for r in reviewed_speeches if r.get('reviewed') is True and r['scene'] in scene_ids]
    if not accepted:
        raise ValueError('Add reviewed speech records for the same reviewed scenes.')
    for r in accepted:
        key = (r['scene'], r['speech_id'])
        if key in speech_ids:
            raise ValueError('Duplicate speech ID within a scene.')
        speech_ids.add(key)
        if not r.get('source_reference') or not r.get('evidence'):
            raise ValueError('Each reviewed speech needs evidence and a source reference.')
        if r['speaker'] not in roster or any(t not in roster for t in r['addressees']):
            raise ValueError('Speech names must occur in the reviewed presence roster. Check names / offstage speech.')
        if r['kind'] not in ('direct', 'group', 'uncertain', 'soliloquy', 'aside'):
            raise ValueError('Use direct, group, uncertain, soliloquy, or aside.')
        if r['kind'] in ('direct', 'group'):
            for target in sorted(set(r['addressees'])):
                if target == r['speaker']:
                    raise ValueError('Do not count self-address as an interpersonal tie.')
                address_rows.append({'Source': r['speaker'], 'Target': target,
                    'scene': r['scene'], 'speech_id': r['speech_id'],
                    'source_reference': r['source_reference'], 'evidence': r['evidence']})
    reviewed_presence_audit = pd.DataFrame(presence_rows, columns=['Source','Target','scene','source_reference'])
    reviewed_address_audit = pd.DataFrame(address_rows, columns=['Source','Target','scene','speech_id','source_reference','evidence'])
    def counted_edges(frame):
        return frame.groupby(['Source','Target']).size().reset_index(name='Weight')
    df_copresence = counted_edges(reviewed_presence_audit)
    df_dialogue = counted_edges(reviewed_address_audit)
    if df_copresence.empty or df_dialogue.empty:
        raise ValueError('Reviewed data need at least one presence pair and one supported directed address.')
    G = nx.from_pandas_edgelist(df_copresence, 'Source', 'Target', edge_attr='Weight')
    G_dialogue = nx.from_pandas_edgelist(df_dialogue, 'Source', 'Target', edge_attr='Weight', create_using=nx.DiGraph())
    G.add_nodes_from(sorted(roster)); G_dialogue.add_nodes_from(sorted(roster))
    relationship_mode = 'Reviewed physical presence / actual addressees'
    print(f'Reviewed scope: {len(scene_ids)} scenes; {len(accepted)} speech records. Check speech coverage against the source.')
    matrix = pd.DataFrame(0, index=scene_ids, columns=sorted(roster))
    for r in selected_scenes:
        for person in r['present']: matrix.loc[r['scene'], person['character']] = 1
    matrix.index.name = 'Scene'
    display(matrix)
else:
    matrix = pd.crosstab(cue_audit.loc[cue_audit.included, 'scene'],
                         cue_audit.loc[cue_audit.included, 'character']).clip(upper=1)
    print('Preview: scene × speaking-character matrix (not full physical presence).')
    display(matrix.head(12))
print(relationship_mode)
print('Use the automatic networks for exploration; use reviewed data before making presence / attention claims.')


## 3. Directed / undirected × weighted / unweighted
Compare all four views with the same people and positions.
- Reviewed undirected: physical co-presence; weight = shared scenes.
- Reviewed directed: speaker → actual addressee; weight = speeches addressed.
- Automatic mode: co-speakers and consecutive speaker transitions, respectively.
Unweighted views count different partners; weighted views retain frequency. Check the selected mode before interpreting the graphs.


In [ ]:
print(relationship_mode)
# Use a common node set and layout to make the comparison readable.
layout_graph = nx.compose(G, G_dialogue.to_undirected())
DISPLAY_TOP = 20  # Set to None to show all characters. Metrics below use the full network.
comparison_nodes = sorted(layout_graph, key=lambda node: G.degree(node), reverse=True)
if DISPLAY_TOP is not None:
    comparison_nodes = comparison_nodes[:DISPLAY_TOP]
comparison_pos = nx.spring_layout(layout_graph.subgraph(comparison_nodes), seed=42, weight=None, k=0.9)
print(f'Comparison shows {len(comparison_nodes)} of {len(layout_graph)} characters. Tables and removal tests use all characters.')
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
for ax, graph, weighted, title in [
    (axes[0, 0], G, False, 'Undirected · unweighted'),
    (axes[0, 1], G, True, 'Undirected · weighted'),
    (axes[1, 0], G_dialogue, False, 'Directed · unweighted'),
    (axes[1, 1], G_dialogue, True, 'Directed · weighted')]:
    graph = graph.subgraph(comparison_nodes)
    maximum = max((d['Weight'] for a, b, d in graph.edges(data=True)), default=1)
    widths = [0.5 + 5 * d['Weight']/maximum if weighted else 0.7
              for a, b, d in graph.edges(data=True)]
    nx.draw_networkx(graph, comparison_pos, ax=ax, node_size=400, font_size=7,
        node_color='#fed976', edge_color='#999', width=widths,
        arrows=graph.is_directed(), **({'arrowsize': 10} if graph.is_directed() else {}))
    ax.set_title(title, fontsize=11)
    ax.axis('off')
fig.tight_layout()
plt.show()


## 4. What structure do you see?
Look for hubs, clusters, bridges, and peripheral characters. Is there evidence for a hub-and-spoke, core-periphery, or several-community structure? Explain one observation before looking at the rankings. A layout alone does not establish a small-world or power-law network.

**Questions 4–5:** Describe each community and its bridges. What overall topology do you see? Support the description with evidence.


In [ ]:
# === Community Detection ===
groups = nx.community.louvain_communities(G, weight='Weight', seed=42)
partition = {node: cid for cid, members in enumerate(groups) for node in members}
num_communities = max(partition.values()) + 1
cmap = plt.get_cmap('Set2', num_communities)
node_colors = [cmap(partition[n]) for n in G.nodes()]

# === Visualization ===
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.8, seed=42)
edge_weights = [G[u][v]['Weight'] * 0.4 for u, v in G.edges()]

nx.draw_networkx_nodes(G, pos, node_size=700, node_color=node_colors, alpha=0.9, edgecolors='black')
nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.5, edge_color='gray')
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')

plt.title("Visualization 1: Co-Presence Network with Communities", fontsize=14, fontweight='bold')
plt.figtext(0.5, 0.01, 'Color = detected community; line width = shared-scene count. Positions are a layout, not geography.', ha='center', fontsize=9)
plt.axis('off')
plt.tight_layout()
plt.show()

# Print communities
print("\nDetected Communities:")
communities = {}
for node, comm in partition.items():
    communities.setdefault(comm, []).append(node)
for cid, members in sorted(communities.items()):
    print(f"  Community {cid}: {', '.join(members)}")

## 5. Who is important—and in what sense?
- **Degree:** number of different partners. **Strength:** total shared-scene counts across partners.
- **Betweenness:** share of shortest paths passing through a character.
- **Closeness:** short paths to others, adjusted for reachable characters.
- **Eigenvector:** connections to other central characters.

Paths below are unweighted; eigenvector uses shared-scene weights. Larger/darker nodes rank higher within a panel; compare ranks, not the numerical scales of different measures.


In [ ]:
degree = dict(G.degree())
strength = dict(G.degree(weight='Weight'))
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)
eigenvector = nx.eigenvector_centrality(G, weight='Weight', max_iter=5000)
metrics = {'Degree': degree, 'Betweenness': betweenness,
           'Closeness': closeness, 'Eigenvector': eigenvector}
centrality_table = pd.DataFrame(metrics)
centrality_table['Strength'] = pd.Series(strength)
centrality_table.index.name = 'Character'
display(centrality_table.sort_values('Degree', ascending=False).round(3))
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (name, values) in zip(axes.flat, metrics.items()):
    maximum = max(values.values()) or 1
    sizes = [250 + 1400 * values[node] / maximum for node in G]
    nx.draw_networkx(G, pos, ax=ax, node_size=sizes,
        node_color=[values[node] / maximum for node in G], cmap=plt.cm.YlOrRd,
        vmin=0, vmax=1, edge_color='#b8b8b8', font_size=7, edgecolors='#555')
    leaders = sorted(values, key=values.get, reverse=True)[:3]
    ax.set_title(name + '\nTop three: ' + ', '.join(leaders), fontsize=10)
    ax.axis('off')
fig.suptitle('Same network and positions; importance depends on the measure', fontsize=14)
fig.tight_layout()
plt.show()


### Global structure and top characters — questions 1–3
1. What do transitivity and average path length suggest? If disconnected, distinguish the whole network from its largest component.
2. Who are the top three by strength, betweenness, closeness, and eigenvector centrality?
3. Find at least two characters whose rankings differ. Explain using a connection and a scene. Strength counts shared scenes across partners—not screen time.


In [ ]:
components_global = list(nx.connected_components(G))
largest_global = G.subgraph(max(components_global, key=len)).copy()
global_metrics = pd.DataFrame([{
    'Nodes': len(G), 'Edges': G.number_of_edges(), 'Density': nx.density(G),
    'Components': len(components_global), 'Transitivity': nx.transitivity(G),
    'Whole-network average path': nx.average_shortest_path_length(G) if nx.is_connected(G) else np.nan,
    'Largest-component nodes': len(largest_global),
    'Largest-component average path': nx.average_shortest_path_length(largest_global)
}])
display(global_metrics.round(3))
print('Paths use unweighted ties. Missing whole-network average = disconnected, not zero.')
for metric in ['Strength','Betweenness','Closeness','Eigenvector']:
    print(metric)
    display(centrality_table[[metric]].sort_values(metric, ascending=False).head(3))


**Question 7:** Which characters bridge communities? Explain their links and compare betweenness with degree.

Choose two characters whose rankings differ across measures. What makes each important? Use the table and a connection in the graph. Does degree tell the same story as strength?

## 6. Who matters when removed?
Predict before running: choose a well-connected character and a bridge. Remove each separately from the original graph. Who disconnects people, and who mainly removes many direct ties?

This removes a node and its links; it does not simulate rewriting the story or other characters adapting.


In [ ]:
original_nodes = len(G)

def removal_summary(graph):
    components = list(nx.connected_components(graph))
    # Fixed original denominator: removed and disconnected pairs contribute zero.
    # This avoids an apparent gain from simply dropping a hard-to-reach character.
    inverse_distances = sum(1 / distance
        for source, distances in nx.all_pairs_shortest_path_length(graph)
        for target, distance in distances.items() if target != source)
    return {'Nodes left': len(graph), 'Links left': graph.number_of_edges(),
            'Components': len(components),
            'Largest component': max(map(len, components), default=0),
            'Isolates': nx.number_of_isolates(graph),
            'Reach efficiency (original pairs)': inverse_distances / (original_nodes * (original_nodes - 1))
                if original_nodes > 1 else 0}

baseline = removal_summary(G)
rows = []
for character in G:
    remaining = G.copy()
    remaining.remove_node(character)
    result = removal_summary(remaining)
    result['Character removed'] = character
    result['Efficiency loss'] = baseline['Reach efficiency (original pairs)'] - result['Reach efficiency (original pairs)']
    rows.append(result)
removal_table = pd.DataFrame(rows).set_index('Character removed')
print('Original network:')
display(pd.DataFrame([baseline]).round(3))
print('Each row starts over from the original network:')
display(removal_table.sort_values(['Largest component', 'Efficiency loss'], ascending=[True, False]).round(3))

remove_choice = widgets.Dropdown(options=sorted(G), description='Remove:', style={'description_width': 'initial'})
remove_button = widgets.Button(description='Remove and compare', button_style='primary')
remove_output = widgets.Output()
def compare_removal(_):
    character = remove_choice.value
    remaining = G.copy()
    remaining.remove_node(character)
    with remove_output:
        from IPython.display import clear_output
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        for ax, graph, title in [(axes[0], G, 'Original'), (axes[1], remaining, 'Removed: ' + character)]:
            nx.draw_networkx(graph, pos, ax=ax, node_size=450,
                node_color=[cmap(partition[node]) for node in graph],
                edge_color='#aaa', font_size=7, edgecolors='#555')
            ax.set_title(title)
            ax.axis('off')
            ax.set_xlim(axes[0].get_xlim())
            ax.set_ylim(axes[0].get_ylim())
        fig.tight_layout()
        plt.show()
        display(pd.DataFrame([baseline, removal_summary(remaining)], index=['Original', 'After removal']).round(3))
remove_button.on_click(compare_removal)
display(widgets.VBox([remove_choice, remove_button, remove_output]))


Explain one removal result. Did it match your prediction? Distinguish losing the removed character's own connections from breaking connections among the people who remain. Removing a character can split the network without removing the most edges.

## 7. Does the answer depend on what counts as a connection?
Use the slider to exclude weak co-presence ties. Compare with the full network. Are the same hubs and bridges still visible? A low-frequency tie can carry a key scene.

**Question 6:** Which relationships survive filtering? What disappears? Explain how the threshold changes your interpretation of the narrative backbone.


In [ ]:
threshold = widgets.IntSlider(value=0, min=0, max=int(df_copresence.Weight.max()), description='Keep weight >', continuous_update=False, style={'description_width': 'initial'})
filter_button = widgets.Button(description='Filter and compare')
filter_output = widgets.Output()
def compare_filter(_):
    filtered = G.copy()
    filtered.remove_edges_from([(a, b) for a, b, data in G.edges(data=True) if data['Weight'] <= threshold.value])
    with filter_output:
        from IPython.display import clear_output
        clear_output(wait=True)
        plt.figure(figsize=(12, 8))
        nx.draw_networkx(filtered, pos, node_size=450,
            node_color=[cmap(partition[node]) for node in filtered], edge_color='#aaa', font_size=7)
        plt.title(f'Keep shared-scene counts > {threshold.value}; retain isolated characters')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        display(pd.DataFrame([baseline, removal_summary(filtered)], index=['Full network', 'Filtered']).round(3))
filter_button.on_click(compare_filter)
display(widgets.VBox([threshold, filter_button, filter_output]))


## 8. Directed counts: partners versus frequency
In/out-degree count distinct incoming/outgoing partners; in/out-strength count their frequency. With reviewed addressees, these describe who addresses whom. With automatic data, they describe previous/next speakers only. Directed eigenvector below uses incoming links; it does not measure outgoing influence.


In [ ]:
in_degree = dict(G_dialogue.in_degree())
out_degree = dict(G_dialogue.out_degree())
in_strength = dict(G_dialogue.in_degree(weight='Weight'))
out_strength = dict(G_dialogue.out_degree(weight='Weight'))
try:
    eig_d = nx.eigenvector_centrality(G_dialogue, weight='Weight', max_iter=5000)
except nx.PowerIterationFailedConvergence:
    eig_d = {node: np.nan for node in G_dialogue}
    print('Eigenvector did not converge; shown as missing, not zero.')

dialogue_df = pd.DataFrame({
    'Character': list(G_dialogue),
    'In-Degree': [in_degree[n] for n in G_dialogue],
    'Out-Degree': [out_degree[n] for n in G_dialogue],
    'In-Strength': [in_strength[n] for n in G_dialogue],
    'Out-Strength': [out_strength[n] for n in G_dialogue],
    'Dialogue Eigenvector': [eig_d[n] for n in G_dialogue]
})
dialogue_df['In/Out Strength Ratio'] = dialogue_df['In-Strength'].div(
    dialogue_df['Out-Strength'].replace(0, np.nan))
dialogue_df = dialogue_df.sort_values('In-Strength', ascending=False)
display(dialogue_df.round(3))

plt.figure(figsize=(14, 10))
pos_d = nx.spring_layout(G_dialogue, k=0.7, seed=42)
d_weights = [G_dialogue[u][v]['Weight'] * 0.12 for u, v in G_dialogue.edges()]

nx.draw_networkx_nodes(G_dialogue, pos_d, node_size=600, node_color='skyblue', alpha=0.9, edgecolors='black')
nx.draw_networkx_edges(G_dialogue, pos_d, width=d_weights, alpha=0.6, edge_color='darkblue',
                       arrowstyle='->', arrowsize=15, connectionstyle='arc3,rad=0.1')
nx.draw_networkx_labels(G_dialogue, pos_d, font_size=8, font_weight='bold')

plt.title("Directed network — check relationship_mode for arrow meaning", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

### Directed patterns — questions 8–10
Use **actual-addressee data** to interpret attention. With automatic data, these are turn-taking patterns only.

8. Identify a receiver (high in-degree), an initiator (high out-degree), a balanced character, and someone connected to prominent receivers. Compare partner counts with frequency.
9. Find an asymmetric pair of arrows. Check a source scene before interpreting power or attention.
10. On the scatterplot, who is above or below the diagonal? Why might this happen?


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
ax.scatter(dialogue_df['Out-Degree'], dialogue_df['In-Degree'], s=90, color='#d95f02')
limit = max(dialogue_df['Out-Degree'].max(), dialogue_df['In-Degree'].max(), 1)
ax.plot([0, limit], [0, limit], '--', color='gray')
for r in dialogue_df.itertuples(index=False, name=None):
    ax.annotate(r[0], (r[2], r[1]), fontsize=7, xytext=(3,3), textcoords='offset points')
ax.set(xlabel='Out-degree: distinct targets', ylabel='In-degree: distinct sources',
       title=relationship_mode + '\nAbove diagonal: more incoming partners; below: more outgoing partners')
plt.tight_layout(); plt.show()

# Same definition of degree for a fair cross-network ranking: undirected projections.
D_projection = G_dialogue.to_undirected()
comparison = pd.DataFrame({'Presence/proxy degree': pd.Series(nx.degree_centrality(G)),
    'Dialogue/proxy degree': pd.Series(nx.degree_centrality(D_projection)),
    'Presence/proxy betweenness': pd.Series(nx.betweenness_centrality(G)),
    'Dialogue/proxy betweenness': pd.Series(nx.betweenness_centrality(D_projection))})
comparison['Presence rank'] = comparison['Presence/proxy degree'].rank(ascending=False, method='min')
comparison['Dialogue rank'] = comparison['Dialogue/proxy degree'].rank(ascending=False, method='min')
comparison['Rank difference'] = comparison['Presence rank'] - comparison['Dialogue rank']
comparison.index.name = 'Character'
display(comparison.sort_values('Rank difference').round(3))
print('Positive rank difference: ranks higher in the directed network’s undirected projection.')
# Weighted eigenvector comparisons use summed reciprocal weights in the projection.
weighted_projection = nx.Graph(); weighted_projection.add_nodes_from(G_dialogue)
for a,b,d in G_dialogue.edges(data=True):
    previous = weighted_projection.get_edge_data(a,b,{}).get('Weight',0)
    weighted_projection.add_edge(a,b,Weight=previous+d['Weight'])
try:
    comparison['Dialogue projection eigenvector'] = pd.Series(nx.eigenvector_centrality(weighted_projection, weight='Weight', max_iter=5000))
except nx.PowerIterationFailedConvergence:
    comparison['Dialogue projection eigenvector'] = np.nan
comparison['Presence eigenvector'] = pd.Series(eigenvector)
display(comparison[['Presence eigenvector','Dialogue projection eigenvector']].round(3))

fig, axes = plt.subplots(1,2,figsize=(18,8))
for graph, ax, title in [(G,axes[0],'Presence / co-speaker network'),(D_projection,axes[1],'Dialogue / transition projection')]:
    nx.draw_networkx(graph.subgraph(comparison_nodes),comparison_pos,ax=ax,node_size=350,font_size=7,node_color='#fdae6b',edge_color='#bbb')
    ax.set_title(title); ax.axis('off')
plt.suptitle('Same displayed characters and positions; rankings above use every node')
plt.tight_layout(); plt.show()


## Compare the two models
Who becomes more central in one network? Who is present but relatively quiet? Does a bridge remain a bridge in dialogue? Use rank differences, the shared layout, and scenes—not the picture alone.

**Mentioned but absent:** name an influential offstage figure. How does excluding mentions change the story? A mention network is a third model; do not silently mix it into presence or dialogue.

## Research question and final synthesis
Choose a question about a bridge, an asymmetric relationship, presence versus speech, absent influence, or network structure.

**Questions 11–13: build three arguments.**
11. **Topology:** make a claim about communities, hubs, or the network's overall structure.
12. **Metric divergence:** explain how two measures reveal different roles.
13. **Asymmetry:** explain a directed relationship using incoming/outgoing ties.

For each: give a number, identify visual evidence, and check a specific scene. An edge records the chosen relationship, not whether it is friendly or hostile. A small-world or power-law claim needs more evidence than a familiar-looking graph.

### Write-up and submission
- State your research question, source, scene rule, relationship definitions, aliases, and verification.
- Develop **2–3 findings** with metrics, figures, and textual evidence. Return to your predictions.
- Discuss limits: absent figures, tone, scene boundaries, temporal change, extraction uncertainty, and any omitted scenes.
- Save your notebook, both edge lists, review/audit files, and a PDF write-up. Include the five views: communities, filtered network, betweenness, directed network, and in/out scatter; add the cross-network comparison and a centrality table.

This restores last year's analysis preparation; the current mini-project assignment page sets the official deadline and submission location.

Original materials: [Session 1](https://github.com/KatherineElkins/networkx_miniproject_3_programming-humanity_2025/blob/main/Session1_Network_Analysis_Handout.docx.pdf) · [Session 2](https://github.com/KatherineElkins/networkx_miniproject_3_programming-humanity_2025/blob/main/Session2_Network_Analysis_Handout.docx.pdf) · [2025 rubric](https://github.com/KatherineElkins/networkx_miniproject_3_programming-humanity_2025/blob/main/IPHS200_Network_Analysis_Rubric.md).


## Export your evidence
Save the edge lists **and** the audit files. Explain your scene rule, aliases, exclusions, and the PDF page references for checked edges. The source hash identifies the exact file counted. The notebook does not claim to detect every character present or infer addressees.


In [ ]:
# Generated from the loaded screenplay, not hand-entered demonstration counts.
output_dir = Path('screenplay_network_outputs_reviewed' if USE_REVIEWED else 'screenplay_network_outputs_automatic')
output_dir.mkdir(exist_ok=True)
outputs = {'automatic_shared_scene_edges.csv': count_networks(cues, aliases, [aliases.get(x,x) for x in include_box.value])[0],
           'automatic_turn_transition_edges.csv': count_networks(cues, aliases, [aliases.get(x,x) for x in include_box.value])[1],
           'shared_scene_audit.csv': co_audit, 'turn_transition_audit.csv': turn_audit,
           'speaker_cue_audit.csv': cue_audit, 'scene_headings.csv': scenes}
for filename, frame in outputs.items():
    frame.to_csv(output_dir / filename, index=False)
metadata = {'source_name': source_name, 'sha256': hashlib.sha256(source_bytes).hexdigest(),
    'extra_headings': extra_headings, 'aliases': aliases, 'included_speaker_labels': list(include_box.value),
    'cooccurrence_rule': 'one count per scene per pair of selected non-voice-over speakers',
    'directed_rule': 'consecutive different speaker cues in the same scene; no addressee inference',
    'voice_over_excluded': True, 'simultaneous_turn_order': 'not inferred'}
(output_dir / 'method.json').write_text(json.dumps(metadata, indent=2))
with zipfile.ZipFile('screenplay_network_outputs.zip', 'w') as archive:
    for path in output_dir.iterdir():
        archive.write(path, arcname=path.name)
print('Saved screenplay_network_outputs.zip and the individual CSV audit files.')
# In Colab, use the Files sidebar to download the ZIP.

metadata['relationship_mode'] = relationship_mode
metadata['reviewed_scene_ids'] = [r['scene'] for r in reviewed_presence if r.get('reviewed') is True] if USE_REVIEWED else []
metadata['cooccurrence_rule'] = 'reviewed physical presence, one count per pair per scene' if USE_REVIEWED else metadata['cooccurrence_rule']
metadata['directed_rule'] = 'reviewed speaker to actual addressee; one count per speech per supported target' if USE_REVIEWED else metadata['directed_rule']
# Preserve automatic audit files; export active networks under unambiguous names.
df_copresence.to_csv(output_dir / 'active_undirected_edges.csv', index=False)
df_dialogue.to_csv(output_dir / 'active_directed_edges.csv', index=False)
matrix.to_csv(output_dir / 'scene_character_matrix.csv')
comparison.to_csv(output_dir / 'cross_network_comparison.csv')
global_metrics.to_csv(output_dir / 'global_metrics.csv', index=False)
centrality_table.to_csv(output_dir / 'centrality.csv')
dialogue_df.to_csv(output_dir / 'directed_centrality.csv', index=False)
for review_path in [presence_path, speech_path]:
    (output_dir / review_path.name).write_text(review_path.read_text())
if USE_REVIEWED:
    reviewed_presence_audit.to_csv(output_dir / 'reviewed_presence_audit.csv', index=False)
    reviewed_address_audit.to_csv(output_dir / 'reviewed_address_audit.csv', index=False)
(output_dir / 'method.json').write_text(json.dumps(metadata, indent=2))
with zipfile.ZipFile('screenplay_network_outputs.zip', 'w') as archive:
    for path in output_dir.iterdir(): archive.write(path, arcname=path.name)
print('Updated ZIP includes active networks, review records, matrix, metrics, and comparisons.')
